# Titanic Survival Dataset 
**Domain : Disaster Survival**

# Objective of This Project :
**Demostrate professional-level data cleaning skills by taking a deliberately messy dataset and systematically transfroming it into a clean, analysis-redy dataset.**

# Feature Checklist:

* Load dataset and produce a "Data Quality Report": count of nulls per column, duplicate rows. data type issues, value range anomalies

* Missing data handling: choose an appropriate strategy for each column (mean/median imputation, mode imputation, forward fill, or row deletion) and justify each choice in a markdown cell 

* Duplicate removal: identify and duplicate rows; document how many were removed

* Standardisation: normalise inconsistent formatting

* Outlier detection: use IQR method or Z-score to identify outliers in numeric columns; decide and documnet whether to cap, remove, or retain each

* Data type correction: ensure all columns have the correct dtype 

# Import Libraries

In [13]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

# Load dataset and produce a "Data Quality Report":
**count of nulls per column, duplicate rows. data type issues, value range anomalies**

In [14]:
# Loading the dataset
df =pd.read_csv('Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Data Quality Report

**Shape of the dataset :**

In [15]:
df.shape

(891, 12)

**Total Records & Columns in Dataset :**
* Records : 891
* Columns : 12

# 

**Duplicate Rows in Dataset :**

In [16]:
df.duplicated().sum()

np.int64(0)

**Checking the data types of each column :**

In [17]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

**Info about Dataset :**

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


**Value range anomalies :**

In [19]:
# Replace df with your DataFrame variable name if it's different.
checks = {
    "Invalid Survived value": df["Survived"].notna() & ~df["Survived"].isin([0, 1]),
    "Invalid passenger class": df["Pclass"].notna() & ~df["Pclass"].isin([1, 2, 3]),
    "Age outside 0–120": df["Age"].notna() & ~df["Age"].between(0, 120),
    "Negative fare": df["Fare"].notna() & df["Fare"].lt(0),
    "Negative SibSp": df["SibSp"].notna() & df["SibSp"].lt(0),
    "Negative Parch": df["Parch"].notna() & df["Parch"].lt(0),
    "Invalid Sex": df["Sex"].notna() & ~df["Sex"].isin(["male", "female"]),
    "Invalid Embarked": df["Embarked"].notna() & ~df["Embarked"].isin(["C", "Q", "S"]),
    "Fractional SibSp": df["SibSp"].notna() & (df["SibSp"] % 1 != 0),
    "Fractional Parch": df["Parch"].notna() & (df["Parch"] % 1 != 0),
}

# Collect all anomaly reasons for each row.
reasons = pd.Series("", index=df.index, dtype="object")

for reason, condition in checks.items():
    mask = condition.fillna(False)
    reasons.loc[mask] = reasons.loc[mask].apply(
        lambda existing: f"{existing}; {reason}" if existing else reason
    )

anomalies = df.loc[reasons.ne("")].copy()
anomalies["Anomaly"] = reasons.loc[reasons.ne("")]

print(anomalies)

Empty DataFrame
Columns: [PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, Anomaly]
Index: []


**Total Null Values by Column :**

In [20]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

# Missing data handling :

In [21]:
df['Age']   = df['Age'].fillna(df['Age'].median())
df['Cabin'] = df['Cabin'].fillna("Unknown")
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

### Justification for Handling Missing Values

- **Age — Median:** Missing ages were replaced with the median because it is less affected by unusually high or low ages than the mean.
- **Cabin — "Unknown":** Missing cabin values were labeled `"Unknown"` to retain those passenger records while making the missing information explicit.
- **Embarked — Mode:** Missing embarkation ports were replaced with the most frequent value because `Embarked` is categorical.

# Duplicate Removal :

In [22]:
print("Total Duplicates:", df.duplicated().sum())

Total Duplicates: 0


### Duplicate Record Check

The dataset was checked for duplicate rows, and none were found. Therefore, no records were removed, and all rows were retained for the next steps.

# Standardisation :

In [23]:
columns_to_scale = ["Age", "Fare"]

scaler = StandardScaler()
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

# Outlier detection :

In [24]:
for col in df.select_dtypes(include=[np.number]).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)
print(f"{col}: {mask.sum()} possible outliers (bounds: {lower:.2f}–{upper:.2f})")
print(df.loc[mask, ["PassengerId", col]])

Fare: 116 possible outliers (bounds: -1.19–0.67)
     PassengerId      Fare
1              2  0.786845
27            28  4.647001
31            32  2.301729
34            35  1.006062
52            53  0.896497
..           ...       ...
846          847  0.751946
849          850  1.145664
856          857  2.671118
863          864  0.751946
879          880  1.025945

[116 rows x 2 columns]
